In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from sklearn.cluster import AgglomerativeClustering
from scipy.spatial.distance import pdist, squareform
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time
import logging
import os
import category_encoders as ce
# Force CPU Device
DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

In [2]:
path = "merged_data_v2.csv"
drop_cols = [
    'Assessment_Effective_Date',
    'Submitted_HIPPS_Code',
    'Facility_Internal_ID',
    'Agency_Medicare_Number',
    'COUNTY_NAME',
    'COUNTYFIPS',
    'ever_deceased',
    'has_diabetes',
    'has_heart_failure',
    'has_hypertension', 'weight'
]

df =pd.read_csv(path,low_memory=False,nrows=100000).drop(drop_cols, axis=1).reset_index(drop=True) #
df

In [3]:
print(f"Complete cohort shape: {df.shape}")
print(f"Number of patients: {df.Beneficiary_ID.nunique()}")
df.columns.to_list()

In [4]:
cols_to_drop = ['ever_readmitted','Beneficiary_ID']

X = df.drop(columns=cols_to_drop, errors='ignore').copy()
y = df['ever_readmitted'].values

processed_dfs = []

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

In [5]:
def cluster_icd_column(series, n_clusters=70, analyze=True):
    """
    Clusters ICD codes. 
    If analyze=True, it runs Silhouette Analysis to find and use the best K.
    """
    from scipy.spatial.distance import pdist, squareform
    from sklearn.cluster import AgglomerativeClustering
    from sklearn.metrics import silhouette_score

    if series is None: return pd.Series([])
    valid_series = series.dropna().astype(str)
    unique_codes = valid_series.unique()
    
    # Pad strings to max length
    max_len = max(len(c) for c in unique_codes)
    def vectorize(s):
        s = s.ljust(max_len)
        return [ord(c) for c in s]
    
    try:
        # 1. Compute Distance Matrix
        X_str = np.array([vectorize(c) for c in unique_codes])
        dist_matrix = squareform(pdist(X_str, metric='hamming'))
        
        # 2. Analysis Mode (Auto-tune K)
        if analyze:
            print("\n    [Auto-Tuning] Analyzing optimal cluster count (Silhouette)... ")
            possible_k = [k for k in range(20, 151, 10)] # 20, 30, ... 150
            best_k = n_clusters
            best_score = -1
            
            for k in possible_k:
                if k >= len(unique_codes): break
                # Use precomputed distance matrix for efficiency
                model = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
                labels = model.fit_predict(dist_matrix)
                
                score = silhouette_score(dist_matrix, labels, metric='precomputed')
                # print(f"      k={k}: Score={score:.4f}")
                
                if score > best_score:
                    best_score = score
                    best_k = k
            
            print(f"    -> Optimal K found: {best_k} (Score: {best_score:.4f})")
            n_clusters = best_k

        # 3. Final Clustering
        if len(unique_codes) <= n_clusters:
            return series
        
        print(f"  -> Clustering {len(unique_codes)} codes into {n_clusters} clusters... ", end=' ')
        clustering = AgglomerativeClustering(n_clusters=n_clusters, metric='precomputed', linkage='average')
        labels = clustering.fit_predict(dist_matrix)
        mapping = {code: f"ICD_Cluster_{label}" for code, label in zip(unique_codes, labels)}
        print("Done.")
        return series.map(mapping).fillna(series)
        
    except Exception as e:
        print(f"-> Clustering failed: {e}. Returning original.")
        return series

In [6]:
# 1. Process Numeric
if num_cols:
    print(f"Processing {len(num_cols)} Numeric columns (Mean Imputation)...")
    imp = SimpleImputer(strategy='mean')
    X_num = pd.DataFrame(imp.fit_transform(X[num_cols]), columns=num_cols)
    processed_dfs.append(X_num)

# 2. Process Categorical
for col in cat_cols:
    series = X[col].copy()
    n_unique = series.nunique()
    
    print(f"Processing Categorical '{col}' (Unique: {n_unique})...", end=' ')
    
    # Check if ICD column (refined based on user input)
    is_icd = False
    c_lower = col.lower()
    # Strict check first, then broad
    if 'icd_10_c_m' in c_lower: 
        is_icd = True
    elif ('icd' in c_lower or 'dx' in c_lower or 'diagnosis' in c_lower) and n_unique > 100:
         # Broad check only if cardinality is high enough to warrant it
         is_icd = True
    
    if is_icd:
        print("-> Detected as ICD column.")
        series = cluster_icd_column(series, n_clusters=70)
        if series is None: 
             print(" -> Warning: Clustering returned None. Using original.")
             series = X[col].copy()
        n_unique = series.nunique()
        print(f"  -> Clustered into {n_unique} groups.", end=' ')
    
    # Encode (and drop original)
    if n_unique > 55:
        print("-> Using Binary Encoding (>55)...")
        if ce:
            # BinaryEncoder replaces the column with col_0, col_1 etc.
            enc = ce.BinaryEncoder(cols=[col])
            res = enc.fit_transform(pd.DataFrame({col: series}))
            processed_dfs.append(res)
        else:
            print("-> Fallback to Ordinal (ce missing).")
            # Replaces with integer codes
            res = pd.DataFrame({col: series.astype('category').cat.codes})
            res[res==-1] = 0 
            processed_dfs.append(res)
    else:
        print("-> Using OneHot Encoding (<=55)...")
        # get_dummies returns only new columns
        res = pd.get_dummies(series, prefix=col, drop_first=False)
        processed_dfs.append(res)
        
# Combine all processed parts
# Note: original 'cat_cols' are NOT in processed_dfs, only their encoded versions.
if processed_dfs:
    X_final = pd.concat(processed_dfs, axis=1)
else:
    X_final = pd.DataFrame()
    
# Cleanup names
X_final.columns = [str(c).replace('[','').replace(']','').replace('<','') for c in X_final.columns]
X_vals = X_final.values

In [7]:
X_final.columns.to_list()

In [8]:
print("\nBalancing Data...")
class_counts = pd.Series(y).value_counts()
if len(class_counts) < 2:
    X_bal, y_bal = X_vals, y
else:
    rus = RandomUnderSampler(random_state=42)
    X_bal, y_bal = rus.fit_resample(X_vals, y)
        
print("Splitting and Scaling...")
X_train, X_test, y_train, y_test = train_test_split(X_bal, y_bal, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

In [9]:
print(f"Final Training Data Shape: {X_train_scaled.shape}")
print(f"Final Testing Data Shape: {X_test_scaled.shape}")

In [10]:
class NumpyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class UnivariateFunction(nn.Module):
    def __init__(self, hidden_units=20, dropout_p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_units),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_units, 1)
        )

    def forward(self, x):
        return self.net(x)


class KAN(nn.Module):
    def __init__(self, input_dim, hidden_units=20, dropout_p=0.1):
        super().__init__()
        self.d = input_dim
        self.univariates = nn.ModuleList([
            UnivariateFunction(hidden_units, dropout_p) for _ in range(2 * self.d + 1)
        ])
        self.coeffs = nn.ParameterList([
            nn.Parameter(torch.randn(input_dim)) for _ in range(2 * self.d + 1)
        ])
        self.outer_weights = nn.Parameter(torch.randn(2 * self.d + 1))
        self.outer_bias = nn.Parameter(torch.randn(1))

    def forward(self, x):
        outputs = []
        for i in range(2 * self.d + 1):
            comb = x @ self.coeffs[i]
            comb = comb.unsqueeze(1)
            out = self.univariates[i](comb)
            outputs.append(out.squeeze(1))
        stacked = torch.stack(outputs, dim=1)
        return (stacked @ self.outer_weights + self.outer_bias).squeeze(-1)

In [11]:
def train_evaluate_kan(X_train, X_test, y_train, y_test, exposure):
    print(f"\n--- Training KAN for {exposure} ---")
    input_dim = X_train.shape[1]
    
    # Hyperparams
    batch_size = 256
    epochs = 10
    lr = 1e-3

    # DataLoaders
    train_loader = DataLoader(NumpyDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(NumpyDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

    # Init Model
    model = KAN(input_dim=input_dim).to(DEVICE)
    crit = nn.BCEWithLogitsLoss()
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_loss = float('inf')
    
    # Loop
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(Xb).squeeze()
            loss = crit(out, yb)
            loss.backward()
            opt.step()
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                out = model(Xb).squeeze()
                loss = crit(out, yb)
                val_loss += loss.item()
                
        if (epoch+1) % 1 == 0:
             print(f"Epoch {epoch+1}: Train Loss {train_loss/len(train_loader):.4f}, Val Loss {val_loss/len(test_loader):.4f}")
             
        if val_loss < best_val_loss:
            best_val_loss = val_loss

    # Final Eval
    model.eval()
    all_probs = []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb = Xb.to(DEVICE)
            out = model(Xb).squeeze()
            probs = torch.sigmoid(out).cpu().numpy()
            all_probs.extend(probs)
            
    y_prob = np.array(all_probs)
    y_pred = (y_prob >= 0.5).astype(int)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2,2) else (0,0,0,0)

    print(f"Finished {exposure}. Acc: {acc:.4f}, F1: {f1:.4f}")
    print(f"  Prec: {prec:.4f}, Rec: {rec:.4f}")
    print(f"  TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")
    
    return {
        'Level': exposure, 
        'Accuracy': acc, 
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn
    }

In [ ]:
results = []

res = train_evaluate_kan(X_train_scaled, X_test_scaled, y_train, y_test, 'Complete_Cohort')
results.append(res)

In [ ]:
results_df = pd.DataFrame(results)
print("\nFinal Results:")
print(results_df)

In [ ]:
class M3Dataset(Dataset):
    def __init__(self, X_micro, X_meso, X_macro, y):
        self.X_micro = torch.as_tensor(X_micro, dtype=torch.float32)
        self.X_meso = torch.as_tensor(X_meso, dtype=torch.float32)
        self.X_macro = torch.as_tensor(X_macro, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        return self.X_micro[idx], self.X_meso[idx], self.X_macro[idx], self.y[idx]

class M3HKAN(nn.Module):
    def __init__(self, dim_micro, dim_meso, dim_macro, output_targets=['primary']):
        super(M3HKAN, self).__init__()
        
        # Micro Branch (Clinical)
        self.micro_net = nn.Sequential(
            nn.Linear(dim_micro, 64), nn.SiLU(),
            nn.Linear(64, 32), nn.SiLU()
        )
        
        # Meso Branch (SDoH)
        dim_meso = max(1, dim_meso) # Safety layer if 0 features
        self.meso_net = nn.Sequential(
            nn.Linear(dim_meso, 32), nn.SiLU(),
            nn.Linear(32, 16), nn.SiLU()
        )
        
        # Macro Branch (Census)
        dim_macro = max(1, dim_macro)
        self.macro_net = nn.Sequential(
            nn.Linear(dim_macro, 32), nn.SiLU(),
            nn.Linear(32, 16), nn.SiLU()
        )
        
        # Fusion
        fusion_dim = 32 + 16 + 16
        self.fusion_net = nn.Sequential(
            nn.Linear(fusion_dim, 64), nn.SiLU(),
            nn.Dropout(0.2), 
            nn.Linear(64, 32), nn.SiLU()
        )
        
        # Head
        self.heads = nn.ModuleDict()
        for target in output_targets:
            self.heads[target] = nn.Linear(32, 1) 
            
    def forward(self, x_micro, x_meso, x_macro):
        h_micro = self.micro_net(x_micro)
        h_meso = self.meso_net(x_meso)
        h_macro = self.macro_net(x_macro)
        
        h_fused = torch.cat([h_micro, h_meso, h_macro], dim=1)
        z = self.fusion_net(h_fused)
        
        outputs = {}
        for name, head in self.heads.items():
            outputs[name] = head(z)
            
        return outputs

In [ ]:
def get_m3_feature_groups(all_features):
    """
    Returns (micro, meso, macro) lists based on the provided feature names.
    """
    # Ensure all features are strings to prevent hashing errors with numpy arrays
    all_features_str = [str(f) for f in all_features]
    feats = set(all_features_str)

    # Helper to check existence to avoid KeyErrors
    def valid(names):
        return [f for f in names if f in feats]
    
    MICRO_FEATURES = valid([
        # Demographics
        'Age', 'Gender_Female', 'Gender_Male',
        'American_Indian_or_Alaska_Native', 'Asian',
        'Black_or_African_American', 'Hispanic_or_Latino',
        'Native_Hawiian_or_Pacific_Islander', 'White',

        # Clinical utilization
        'Days_Cared_For', 'BMI',
        'BMI_Category_Normal weight',
        'BMI_Category_Obese-Class1',
        'BMI_Category_Obese-Class2',
        'BMI_Category_Obese-Class3',
        'BMI_Category_Overweight',
        'BMI_Category_Underweight'
        # Discipline
        'ByDiscipline_OT', 'ByDiscipline_PT',
        'ByDiscipline_RN', 'ByDiscipline_SLP/ST',

        # Comorbidities (Charlson / Elixhauser-style)
        'aids', 'alcohol', 'carit', 'chf', 'coag', 'cpd', 'dane', 'depre', 'drug', 'fed',
        'hypc','hypothy', 'hypunc', 'ld', 'lymph', 'metacanc', 'obes', 'ond',
        'para','pcd', 'psycho', 'pvd','rf','rheumd','solidtum','valv','wloss','blane',
        'diabc','diabunc','pud','comorbidity_score','age_adj_comorbidity_score',
        'ami','canc','cevd','copd','dementia','hp','mld','rend','diab','diabwc','msld',
        'hfrs','frail','impaired_hearing','impaired_speech','impaired_vision','ld_asd'] + [
        # ICD clusters (diagnosis representations)
        c for c in all_features_str if c.startswith('Primary_Diagnosis_ICD')
    ] + [
        c for c in all_features_str if c.startswith('Other_Diagnosis_Code')
    ]  + [
        c for c in all_features_str if c.startswith('ICD_Section_ICD')
    ] + [
        c for c in all_features_str if c.startswith('ICD_Range_ICD')
    ])

    MESO_FEATURES = valid([
        # Urban / Rural composition
        'COUNTY_NAME',
        'POP_URB', 'POPPCT_URB',
        'POP_RUR', 'POPPCT_RUR',

        'ACS_PCT_BACHELOR_DGR',
        'ACS_PCT_COLLEGE_ASSOCIATE_DGR',
        'ACS_PCT_GRADUATE_DGR',
        'ACS_PCT_HS_GRADUATE',
        'ACS_PCT_LT_HS',
        'ACS_PCT_NO_WORK_NO_SCHL_16_19',
        'ACS_PCT_POSTHS_ED',
        'ACS_PCT_VET_BACHELOR',
        'ACS_PCT_VET_COLLEGE',
        'ACS_PCT_VET_HS',
        'ACS_PCT_HH_LIMIT_ENGLISH',
        'ACS_PCT_HH_BROADBAND',
        'ACS_PCT_HH_BROADBAND_ONLY',
        'ACS_PCT_HH_CELLULAR',
        'ACS_PCT_HH_CELLULAR_ONLY',
        'ACS_PCT_HH_DIAL_INTERNET_ONLY',
        'ACS_PCT_HH_INTERNET',
        'ACS_PCT_HH_INTERNET_NO_SUBS',
        'ACS_PCT_HH_NO_COMP_DEV',
        'ACS_PCT_HH_NO_INTERNET',
        'ACS_PCT_HH_OTHER_COMP',
        'ACS_PCT_HH_OTHER_COMP_ONLY',
        'ACS_PCT_HH_PC',
        'ACS_PCT_HH_PC_ONLY',
        'ACS_PCT_HH_SAT_INTERNET',
        'ACS_PCT_HH_SMARTPHONE',
        'ACS_PCT_HH_SMARTPHONE_ONLY',
        'ACS_PCT_HH_TABLET',
        'ACS_PCT_HH_TABLET_ONLY',
        'ACS_PCT_CHILDREN_GRANDPARENT',
        'ACS_PCT_CHILD_1FAM',
        'ACS_PCT_GRANDP_NO_RESPS',
        'ACS_PCT_GRANDP_RESPS_NO_P',
        'ACS_PCT_GRANDP_RESPS_P',
        'ACS_PCT_HH_1PERS',
        'ACS_PCT_HH_ABOVE65',
        'ACS_PCT_HH_ALONE_ABOVE65',
        'ACS_PCT_HH_KID_1PRNT',
        'ACS_TOT_GRANDCHILDREN_GP',
        'ACS_PCT_HEALTH_INC_138_199',
        'ACS_PCT_HEALTH_INC_200_399',
        'ACS_PCT_HEALTH_INC_ABOVE400',
        'ACS_PCT_HEALTH_INC_BELOW137',
        'ACS_PCT_HH_1FAM_FOOD_STMP',
        'ACS_PCT_HH_FOOD_STMP_BLW_POV',
        'ACS_PCT_HH_NO_FD_STMP_BLW_POV',
        'ACS_PCT_HH_PUB_ASSIST',
        'ACS_PCT_INC50',
        'ACS_PCT_INC50_ABOVE65',
        'ACS_PCT_NONVET_POV_18_64',
        'ACS_PCT_PERSON_INC_100_124',
        'ACS_PCT_PERSON_INC_125_199',
        'ACS_PCT_PERSON_INC_ABOVE200',
        'ACS_PCT_PERSON_INC_BELOW99',
        'ACS_PCT_POV_AIAN',
        'ACS_PCT_POV_ASIAN',
        'ACS_PCT_POV_BLACK',
        'ACS_PCT_POV_HISPANIC',
        'ACS_PCT_POV_MULTI',
        'ACS_PCT_POV_NHPI',
        'ACS_PCT_POV_OTHER',
        'ACS_PCT_POV_WHITE',
        'ACS_PCT_VET_POV_18_64',
        'ACS_TOT_POP_POV
    ])

    MACRO_FEATURES = [
        # Agency / system identifiers
        c for c in all_features_str if c.startswith('Agency_Medicare_Number')
    ] + [
        # Payment / case-mix
        c for c in all_features_str if c.startswith('Submitted_HIPPS')
    ]
    
    return MICRO_FEATURES, MESO_FEATURES, MACRO_FEATURES

In [31]:
def split_features_by_level(feature_names):
    """
    Splits features into Micro (Clinical), Meso (SDoH), and Macro (Policy/System) 
    based on the user's detailed schema.
    """
    # FORCE STRING CONVERSION IMMEDIATELY
    feature_names = [str(f) for f in feature_names]
    
    # Get precise lists from the user-defined helper
    micro_list, meso_list, macro_list = get_m3_feature_groups(feature_names)
    
    # Convert to sets for O(1) lookup
    micro_set = set(micro_list)
    meso_set = set(meso_list)
    macro_set = set(macro_list)
    
    micro_idxs = []
    meso_idxs = []
    macro_idxs = []
    
    for idx, f in enumerate(feature_names):
        # f is guaranteed string now
        if f in micro_set:
            micro_idxs.append(idx)
        elif f in meso_set:
            meso_idxs.append(idx)
        elif f in macro_set:
            macro_idxs.append(idx)
        else:
            # Fallback: if not in any specific list, assume Micro (Clinical)
            micro_idxs.append(idx)
            
    return micro_idxs, meso_idxs, macro_idxs

In [32]:
def train_evaluate_m3(X_tr, X_te, y_tr, y_te, exposure):
    
    # 1. Split Features
    print("Splitting features into levels...")
    # FIX: Use global column names, not the numpy array
    if 'X_final' in globals() and hasattr(X_final, 'columns'):
        feats = X_final.columns.tolist()
    else:
        # Fallback if X_final not found, though unlikely in this notebook
        print("Warning: X_final global not found, using indices as features")
        feats = [str(i) for i in range(X_tr.shape[1])]
        
    mic_idx, mes_idx, mac_idx = split_features_by_level(feats)
    print(f"  Micro: {len(mic_idx)}, Meso: {len(mes_idx)}, Macro: {len(mac_idx)}")
    
    def get_parts(X, indices):
        if not indices: return np.zeros((len(X), 1))
        return X[:, indices]

    X_mic_tr, X_mes_tr, X_mac_tr = get_parts(X_tr, mic_idx), get_parts(X_tr, mes_idx), get_parts(X_tr, mac_idx)
    X_mic_te, X_mes_te, X_mac_te = get_parts(X_te, mic_idx), get_parts(X_te, mes_idx), get_parts(X_te, mac_idx)

    # 2. Setup
    train_ds = M3Dataset(X_mic_tr, X_mes_tr, X_mac_tr, y_tr)
    test_ds = M3Dataset(X_mic_te, X_mes_te, X_mac_te, y_te)
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

    model = M3HKAN(
        dim_micro=X_mic_tr.shape[1],
        dim_meso=X_mes_tr.shape[1],
        dim_macro=X_mac_tr.shape[1]
    ).to(DEVICE)
    
    crit = nn.BCEWithLogitsLoss()
    opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # 3. Training Loop
    print(f"\n--- Training M3 HKAN for {exposure} ---")
    best_val_loss = float('inf')
    
    for epoch in range(10):
        model.train()
        train_loss = 0
        for mic, mes, mac, y in train_loader:
            mic, mes, mac, y = mic.to(DEVICE), mes.to(DEVICE), mac.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            out = model(mic, mes, mac)['primary'].squeeze()
            loss = crit(out, y)
            loss.backward()
            opt.step()
            train_loss += loss.item()
            
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for mic, mes, mac, y in test_loader:
                mic, mes, mac, y = mic.to(DEVICE), mes.to(DEVICE), mac.to(DEVICE), y.to(DEVICE)
                out = model(mic, mes, mac)['primary'].squeeze()
                loss = crit(out, y)
                val_loss += loss.item()
        
        print(f"Epoch {epoch+1}: TrLoss {train_loss/len(train_loader):.4f}, ValLoss {val_loss/len(test_loader):.4f}")

    # 4. Final Metrics
    model.eval()
    all_probs = []
    with torch.no_grad():
        for mic, mes, mac, y in test_loader:
            mic, mes, mac = mic.to(DEVICE), mes.to(DEVICE), mac.to(DEVICE)
            out = model(mic, mes, mac)['primary'].squeeze()
            probs = torch.sigmoid(out).cpu().numpy()
            all_probs.extend(probs)
            
    y_pred = (np.array(all_probs) >= 0.5).astype(int)
    
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    cm = confusion_matrix(y_te, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2,2) else (0,0,0,0)
    
    print(f"\nResults ({exposure}): Acc={acc:.4f}, F1={f1:.4f}")
    print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}")
    return {'Level': exposure, 'Acc': acc, 'F1': f1, 'Prec': prec, 'Rec': rec}

In [33]:
# Main Run
try:
    # Ensure function is redefined in memory if running sequentially
    res = train_evaluate_m3(X_train_scaled, X_test_scaled, y_train, y_test, exposure= 'Complete_Cohort')
    print(pd.DataFrame([res]))
except Exception as e:
    import traceback
    traceback.print_exc()